| SOURCE_TABLE_NAME | SOURCE_COLUMN_NAME | REPORTING_TABLE_NAME | REPORTING_COLUMN_NAME | TRANSFORMATION RULE | CONDITIONS |
| --- | --- |--- | --- |--- |--- |
| silver.daily_pricing_silver	| state_name	|***REPORTING_DIM_STATE_GOLD***	|***STATE_NAME***	| Select Unique state_name Values | Identify New/Changed Records From the Source Table . Use ***lakehouse_updated_date Column*** in source table to Identify New/Changed Records|
| DERIVED	| DERIVED	|***REPORTING_DIM_STATE_GOLD***	| ***STATE_ID***	| Generate Running Sequnece ID For Each Unique state_name Values | 1. Make Sure No Duplicate State_Name Values Loaded 2.  When loading new State_Name values in subsquent run , STATE_ID values need to be generated on top of existing MAX(STATE_ID)   |
| DERIVED	| DERIVED	|***REPORTING_DIM_STATE_GOLD***	| ***lakehouse_inserted_date***	| Load current_timestamp() | |
| DERIVED	| DERIVED	|***REPORTING_DIM_STATE_GOLD***	| ***lakehouse_updated_date***	| Load current_timestamp() | |

_**gold.reporting_dim_state_gold**_

In [0]:
%sql

USE CATALOG adb_rtp;

CREATE or replace table silver.reporting_dim_state_stage_1 as
SELECT DISTINCT state_name
FROM silver.daily_pricing_silver
where source_file_load_date > (select NVl(max(processed_file_table_date),"2023-05-01")from processrunlogs.deltalakehouse_process_runs where process_name = 'reporting_dimension_table_load' and process_status = 'complete')

--creating staging or temp tables for intermediate transformations, inter tables in silver layer

In [0]:
%sql
USE CATALOG adb_rtp;

insert into gold.reporting_dim_state_gold
Select state_name, row_number() over(order by state_name) as STATE_ID, current_timestamp(), current_timestamp() from silver.reporting_dim_state_stage_1;

--Create an Identity column using row_number to create a unique id for each record

In [0]:
%sql

create or replace table silver.reporting_dim_state_stage_2 as 
Select silver_dim.state_name, row_number() over(order by silver_dim.state_name) as STATE_ID, current_timestamp() lakehouse_inserted_date, current_timestamp() lakehouse_updated_date from silver.reporting_dim_state_stage_1 silver_dim
left outer join gold.reporting_dim_state_gold gold_dim
on silver_dim.state_name = gold_dim.state_name 
where gold_dim.STATE_NAME is null


--Change Data Capture
--Cant select and insert into a table in the same query, in this case we are checking for missing records in target table so we cant do an insert in the same query

In [0]:
%sql

create or replace table silver.reporting_dim_state_stage_3 
select silverDim.state_name STATE_NAME, silverDim.state_id +prev_max_st_id as STATE_ID, current_timestamp() lakehouse_inserted_date, current_timestamp() lakehouse_updated_date from silver.reporting_dim_state_stage_2 silverDim
cross join (select NVL(max(state_id),0) as prev_max_st_id from gold.reporting_dim_state_gold) goldDim
    
--Create a sub query to handle the mismatching state_id values when identifying the missing values to be inserted

In [0]:
%sql
Insert into gold.reporting_dim_state_gold
SELECT * FROM adb_rtp.silver.reporting_dim_state_stage_3

****

_**gold._reporting_dim_state_gold_**_

In [0]:
%sql

USE CATALOG adb_rtp;

CREATE or replace table silver.reporting_dim_market_stage_1 as
SELECT DISTINCT market_name
FROM silver.daily_pricing_silver
where source_file_load_date > (select NVl(max(processed_file_table_date),"2023-05-01")from processrunlogs.deltalakehouse_process_runs where process_name = 'reporting_dimension_table_load' and process_status = 'complete')

--creating staging or temp tables for intermediate transformations, intermediate tables in silver layer

In [0]:
%sql

create or replace table silver.reporting_dim_market_stage_2 as 
Select silver_dim.market_name, row_number() over(order by silver_dim.market_name) as MARKET_ID, current_timestamp() lakehouse_inserted_date, current_timestamp() lakehouse_updated_date from silver.reporting_dim_market_stage_1 silver_dim
left outer join gold.reporting_dim_market_gold gold_dim
on silver_dim.market_name = gold_dim.market_name 
where gold_dim.MARKET_NAME is null


--Change Data Capture
--Cant select and insert into a table in the same query, in this case we are checking for missing records in target table so we cant do an insert in the same query

In [0]:
%sql

create or replace table silver.reporting_dim_market_stage_3 
select silverDim.market_name MARKET_NAME, silverDim.market_id +prev_max_mt_id as MARKET_ID, current_timestamp() lakehouse_inserted_date, current_timestamp() lakehouse_updated_date from silver.reporting_dim_market_stage_2 silverDim
cross join (select NVL(max(market_id),0) as prev_max_mt_id from gold.reporting_dim_market_gold) goldDim
    
--Create a sub query to handle the mismatching state_id values when identifying the missing values to be inserted

In [0]:
%sql
USE CATALOG adb_rtp;

insert into gold.reporting_dim_market_gold
Select market_name, MARKET_ID, current_timestamp(), current_timestamp() from silver.reporting_dim_market_stage_3;

--Create an Identity column using row_number to create a unique id for each record

**_gold.reporting_dim_variety_gold_**

In [0]:
%sql

USE CATALOG adb_rtp;

CREATE or replace table silver.reporting_dim_variety_stage_1 as
SELECT DISTINCT variety
FROM silver.daily_pricing_silver
where source_file_load_date > (select NVl(max(processed_file_table_date),"2023-05-01")from processrunlogs.deltalakehouse_process_runs where process_name = 'reporting_dimension_table_load' and process_status = 'complete')

--creating staging or temp tables for intermediate transformations, inter tables in silver layer

In [0]:
%sql

create or replace table silver.reporting_dim_variety_stage_2 as 
Select silver_dim.variety, row_number() over(order by silver_dim.variety) as VARIETY_ID, current_timestamp() lakehouse_inserted_date, current_timestamp() lakehouse_updated_date from silver.reporting_dim_variety_stage_1 silver_dim
left outer join gold.reporting_dim_variety_gold gold_dim
on silver_dim.variety = gold_dim.variety 
where gold_dim.VARIETY is null


--Change Data Capture
--Cant select and insert into a table in the same query, in this case we are checking for missing records in target table so we cant do an insert in the same query

In [0]:
%sql

create or replace table silver.reporting_dim_variety_stage_3 
select silverDim.variety VARIETY , silverDim.variety_id +prev_max_variety_id as VARIETY_ID, current_timestamp() lakehouse_inserted_date, current_timestamp() lakehouse_updated_date from silver.reporting_dim_variety_stage_2 silverDim
cross join (select NVL(max(variety_id),0) as prev_max_variety_id from gold.reporting_dim_variety_gold) goldDim
    
--Create a sub query to handle the mismatching state_id values when identifying the missing values to be inserted

In [0]:
%sql
USE CATALOG adb_rtp;

insert into gold.reporting_dim_variety_gold
Select variety, VARIETY_ID, current_timestamp(), current_timestamp() from silver.reporting_dim_variety_stage_3;

--Create an Identity column using row_number to create a unique id for each record

**_gold.reporting_dim_product_gold_**

In [0]:
%sql

USE CATALOG adb_rtp;

CREATE or replace table silver.reporting_dim_product_stage_1 as
SELECT DISTINCT PRODUCT_NAME, PRODUCTGROUP_NAME
FROM silver.daily_pricing_silver
where source_file_load_date > (select NVl(max(processed_file_table_date),"2023-05-01")from processrunlogs.deltalakehouse_process_runs where process_name = 'reporting_dimension_table_load' and process_status = 'complete')

--creating staging or temp tables for intermediate transformations, inter tables in silver layer

In [0]:
%sql

create or replace table silver.reporting_dim_product_stage_2 as 
Select silver_dim.product_name PRODUCT_NAME, silver_dim.productgroup_name PRODUCTGROUP_NAME, row_number() over(order by silver_dim.product_name, silver_dim.productgroup_name) as PRODUCT_ID, current_timestamp() lakehouse_inserted_date, current_timestamp() lakehouse_updated_date from silver.reporting_dim_product_stage_1 silver_dim
left outer join gold.reporting_dim_product_gold gold_dim
on silver_dim.product_name = gold_dim.product_name 
and silver_dim.productgroup_name = gold_dim.productgroup_name 
where gold_dim.PRODUCT_NAME is null;


--Change Data Capture
--Cant select and insert into a table in the same query, in this case we are checking for missing records in target table so we cant do an insert in the same query
--we can just check if one of the columns is null and insert the record

In [0]:
%sql

create or replace table silver.reporting_dim_product_stage_3 
select silverDim.product_name PRODUCT_NAME, silverDim.productgroup_name PRODUCTGROUP_NAME, silverDim.product_id +prev_max_product_id as PRODUCT_ID, current_timestamp() lakehouse_inserted_date, current_timestamp() lakehouse_updated_date from silver.reporting_dim_product_stage_2 silverDim
cross join (select NVL(max(product_id),0) as prev_max_product_id from gold.reporting_dim_product_gold) goldDim
    
--Create a sub query to handle the mismatching state_id values when identifying the missing values to be inserted

In [0]:
%sql
USE CATALOG adb_rtp;

insert into gold.reporting_dim_product_gold
Select  PRODUCTGROUP_NAME,PRODUCT_NAME, PRODUCT_ID, current_timestamp(), current_timestamp() from silver.reporting_dim_product_stage_3;

--Create an Identity column using row_number to create a unique id for each record

In [0]:
%sql
insert into processrunlogs.deltalakehouse_process_runs(process_name, processed_table_datetime, process_status)
select 'reporting_dimension_table_load', max(lakehouse_updated_date),'Completed' from silver.daily_pricing_silver